Create variable about number of conflicts in previous month
<hr>

In [1]:
# import necessary packages
import numpy as np
import pandas as pd
import os

# set working directory
os.chdir("/Users/varvarailyina/hertie/ML/US_conflict_forecasting/")

In [2]:
# load clean conflict data
df_acled_clean = pd.read_csv("./Dataset/ConflictDataClean.csv")

# look at data
df_acled_clean.head()

,ZCTA,event_MY,admin2,fatalities,event_count
0,15001,2022-01,Beaver,0,1
1,15001,2022-02,Beaver,0,0
2,15001,2022-03,Beaver,0,0
3,15001,2022-04,Beaver,0,0
4,15001,2022-05,Beaver,0,0


In [3]:
# load clean data with all PA zip codes and counties
df_all_clean = pd.read_csv("./Dataset/create_acs_conflict/acs_conflict.csv")

# look at data
df_all_clean.head()

,ZCTA,Month,Name,"Population, 2020","# Square Land Miles, 2020","Population per Square Land Mile, 2022",Rural_Urban_Commerical,"Total Population, 2022","Median age, 2022","<18 Years Old, 2022",...,"Asian, Not Hispanic or Latino, 2022","Other Races, Not Hispanic or Latino, 2022","Two or more races, 2022","Hispanic or Latino (Any Race), 2022","Poverty Rate, 2022",ZCTA.1,month,admin2,fatalities,event_count
0,15001.0,2022-01,"Aliquippa, PA",32299.0,60.348429,535.208629,Urban,32237.0,47.5,0.169153,...,0.006359,0.001396,0.04166,0.025313,0.104669,15001.0,2022-01,Beaver,0.0,1.0
1,15001.0,2022-02,"Aliquippa, PA",32299.0,60.348429,535.208629,Urban,32237.0,47.5,0.169153,...,0.006359,0.001396,0.04166,0.025313,0.104669,15001.0,2022-02,Beaver,0.0,0.0
2,15001.0,2022-03,"Aliquippa, PA",32299.0,60.348429,535.208629,Urban,32237.0,47.5,0.169153,...,0.006359,0.001396,0.04166,0.025313,0.104669,15001.0,2022-03,Beaver,0.0,0.0
3,15001.0,2022-04,"Aliquippa, PA",32299.0,60.348429,535.208629,Urban,32237.0,47.5,0.169153,...,0.006359,0.001396,0.04166,0.025313,0.104669,15001.0,2022-04,Beaver,0.0,0.0
4,15001.0,2022-05,"Aliquippa, PA",32299.0,60.348429,535.208629,Urban,32237.0,47.5,0.169153,...,0.006359,0.001396,0.04166,0.025313,0.104669,15001.0,2022-05,Beaver,0.0,0.0


In [4]:
# generate function to create binary variable if conflict in previous month in same zip code

def check_prev_month_conflict(df):
    
    # sort df by month
    df.sort_values(by='Month', inplace=True)
    
    # empty list to store result
    event_count_prev = []

    # for loop per row
    for index, row in df.iterrows():
        
        # vars with current month and year
        current_month = row['Month']
        
        # calculate previous month
        prev_year, prev_month = current_month.split('-')
        prev_month = str(int(prev_month) - 1).zfill(2)
        
        if prev_month == '00':
            prev_year = str(int(prev_year) - 1)
            prev_month = '12'

        prev_month = f"{prev_year}-{prev_month}"

        # check if previous month exists in df
        prev_month_tmp = df[df['Month'] == prev_month]

        # check if previous month had a conflict
        if not prev_month_tmp.empty:
            previous_month_event_num = prev_month_tmp.iloc[0]['event_count']
            event_count_prev.append(previous_month_event_num)

        # if no previous month, assume no conflict
        else:
            event_count_prev.append(0)

    # append result to new var
    df['event_count_prev'] = event_count_prev
    #df.loc[:, 'event_count_prev'] = event_count_prev

    return df

In [10]:
# run function on df
df_out = check_prev_month_conflict(df_all_clean)

In [11]:
# look at zip codes with repeated conflicts
df_out[(df_out['event_count'] > 0) & (df_out['event_count_prev'] > 0)]

,ZCTA,Month,Name,"Population, 2020","# Square Land Miles, 2020","Population per Square Land Mile, 2022",Rural_Urban_Commerical,"Total Population, 2022","Median age, 2022","<18 Years Old, 2022",...,"Other Races, Not Hispanic or Latino, 2022","Two or more races, 2022","Hispanic or Latino (Any Race), 2022","Poverty Rate, 2022",ZCTA.1,month,admin2,fatalities,event_count,event_count_prev
14437,17837.0,2022-02,"Lewisburg, PA",18734.0,56.191610,333.394969,Urban,18670.0,36.7,0.146010,...,0.002035,0.008409,0.065667,0.114748,17837.0,2022-02,Union,0.0,1.0,1.0
21589,19530.0,2022-02,"Kutztown, PA",14830.0,54.605110,271.586301,Rural,14616.0,25.8,0.113506,...,0.000000,0.023810,0.065818,0.110174,19530.0,2022-02,Berks,0.0,1.0,1.0
12745,17331.0,2022-02,"Hanover, PA",55458.0,74.417671,745.226229,Urban,54942.0,41.6,0.219559,...,0.002730,0.024389,0.074060,0.074664,17331.0,2022-02,York,0.0,1.0,1.0
9685,16801.0,2022-02,"State College, PA",42446.0,31.001484,1369.160245,Urban,43307.0,26.3,0.121020,...,0.001847,0.036022,0.034244,0.302329,16801.0,2022-02,Centre,0.0,1.0,1.0
2293,15370.0,2022-02,"Waynesburg, PA",14153.0,148.179072,95.512813,Rural,13886.0,43.5,0.158793,...,0.005545,0.059628,0.023909,0.113571,15370.0,2022-02,Greene,0.0,1.0,1.0
1405,15219.0,2022-02,"Pittsburgh, PA",10949.0,2.200628,4975.398539,Urban,10680.0,32.4,0.220412,...,0.007303,0.043446,0.039232,0.346298,15219.0,2022-02,Allegheny,0.0,10.0,1.0
7765,16335.0,2022-02,"Meadville, PA",27452.0,114.510875,239.732690,Rural,27497.0,42.9,0.189402,...,0.000618,0.034222,0.015129,0.114554,16335.0,2022-02,Crawford,0.0,1.0,1.0
18697,18901.0,2022-02,"Doylestown, PA",28395.0,20.970946,1354.016198,Urban,28311.0,46.7,0.193670,...,0.005051,0.041044,0.026527,0.040835,18901.0,2022-02,Bucks,0.0,1.0,1.0
20029,19107.0,2022-02,"Philadelphia, PA",17600.0,0.544015,32352.025246,Urban,15232.0,31.1,0.055869,...,0.003611,0.051339,0.061909,0.217631,19107.0,2022-02,Philadelphia,0.0,9.0,1.0
15625,18042.0,2022-02,"Easton, PA",44026.0,22.113366,1990.922578,Urban,44818.0,36.7,0.203847,...,0.010398,0.055201,0.202865,0.123315,18042.0,2022-02,Northampton,0.0,1.0,1.0


In [12]:
# check that it worked
df_out[df_out['Name'] == "King Of Prussia, PA"][['ZCTA', 'Month', 'event_count', 'event_count_prev']]

,ZCTA,Month,event_count,event_count_prev
21024,19406.0,2022-01,0.0,0.0
21025,19406.0,2022-02,0.0,1.0
21026,19406.0,2022-03,0.0,0.0
21027,19406.0,2022-04,0.0,0.0
21028,19406.0,2022-05,1.0,0.0
21029,19406.0,2022-06,2.0,1.0
21030,19406.0,2022-07,0.0,0.0
21031,19406.0,2022-08,0.0,0.0
21032,19406.0,2022-09,0.0,0.0
21033,19406.0,2022-10,0.0,0.0


<br>
**when certain that changes are correct, edit the name of the `.csv` file below to overwrite the `acs_conflict.csv`

In [13]:
# order df
df_sorted = df_out.sort_values(by = ['ZCTA', 'Month'])

# save df
df_sorted.to_csv("./Dataset/create_acs_conflict/acs_conflict_varv.csv", index = False)
#df.to_csv("./Dataset/create_acs_conflict/acs_conflict.csv", index = False)